# Train the GuardianEye audio branch

Run this on a GPU runtime. The default backend is the small log-mel CNN. Set `backend: panns` in `configs/audio.yaml` only after you place a local PANNs checkpoint; this notebook does not download one.

The trainer runs Belize to Vietnam, Vietnam to Belize, and both in-domain controls. Validation rows come from the training site. Read `reports/audio/experiments.json` after evaluation and keep the four rows side by side.

Dataset URLs in `configs/data.yaml` are `TODO` until you fill them in.

In [ ]:
!git clone https://github.com/YOUR_USER/guardianeye.git
%cd guardianeye
!pip install -r requirements.txt && pip install -e .

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_audio_data.py

In [ ]:
!python -m guardianeye.audio.train
!python -m guardianeye.eval.audio_eval

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi

repo_id = os.environ["HF_MODEL_REPO"]
api = HfApi()
api.create_repo(repo_id, repo_type="model", exist_ok=True)
for path in sorted(Path("models/audio").glob("*")):
    if path.suffix not in {".pt", ".json"}:
        continue
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=f"audio/{path.name}",
        repo_id=repo_id,
        repo_type="model",
    )
print("uploaded audio checkpoints to", repo_id)